In [0]:
# Notebook to define methods which can be reused

In [0]:
%run "./nt_connection"

In [0]:
%run "./nt_ddl_deltalake"

In [0]:
#Method to Fetch DB2 tables and return the result as a spark dataframe

def getTable(tableName, query=None):
  if query is None:
    query = "{0}.{1}".format(db_schema, tableName)
  else:
    query = query.format(db_schema)
    
  jdbc_df = spark.read.format("jdbc") \
    .option('url', connection_string) \
    .option('driver', 'com.ibm.db2.jcc.DB2Driver') \
    .option('dbtable', query) \
    .option("isolationlevel","READ_UNCOMMITTED") \
    .load()
  return jdbc_df

In [0]:
#Method to insert log data to deltalake log table

import datetime
def udfInsertLogDetails(status_ind,comment_nm):
    
    try:
        dtCurrentDate = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        df_logdetail = spark.createDataFrame(
        [
            (str(varRunId),str(varJobId),varNtName,dtCurrentDate,status_ind,comment_nm),  
        ],
        ["run_id", "job_id", "notebook_nm", "created_dt", "status_ind", "comment_nm"]  
        ) 

        
        df_logdetail.write.format("delta").mode("append").saveAsTable(""+delta_schema+".ms_nt_logdetails")
        return "Success"
    except Exception as e:
        print(str(e))
        raise (e)

In [0]:
#Method to read and write to delta lake table

def getDeltaTable(tableName, query=None):
  if query is None:
    query = "select * from {0}.{1}".format(delta_schema, tableName)
  else:
    query = query.format(delta_schema)
  delta_df = spark.sql(query)
  return delta_df
 
def addDeltaTable(tableName, dataFrame, writemode):
  dataFrame.write.format("delta").mode(writemode).saveAsTable("{0}.{1}".format(delta_schema, tableName))

def dropDeltaTable(tableName, query=None):
  if query is None:
    query = "drop table if exists {0}.{1}".format(delta_schema, tableName)
  else:
    query = query.format(delta_schema)
  delta_df = spark.sql(query)
  return delta_df

def addDeltaTablewithSchema(tableName, dataFrame, writemode,overwriteSchema ):
  dataFrame.write.format("delta").mode(writemode).option("overwriteSchema", overwriteSchema).saveAsTable("{0}.{1}".format(delta_schema, tableName))

In [0]:
def executeDeltaTable(query):
    try:
        query = query.format(delta_schema)
        delta_df = spark.sql(query)
        return delta_df
    except Exception as e:
        print(str(e))
        raise (e)

In [0]:
#Method to read data from EDM-Snowflake and return as a dataframe

# def getEdmTable(query):    
  
#   query = query.format(edmDatabase,edmSchema,edmDatabase_edw,edmSchema_edw,edmDatabase_lwsn)
  
  
#   df = spark.read.format(SNOWFLAKE_SOURCE_NAME) \
#     .options(**sf_snow_pass_Options) \
#     .option("query", query) \
#     .load()
  
#   return df

In [0]:
#Method to read data from EDW-Snowflake and return as a dataframe

# def getEdwTable(query):
#   jdbc_df = spark.read.format(SNOWFLAKE_SOURCE_NAME) \
#     .options(**sf_edw_snow_pass_Options) \
#     .option("query", query) \
#     .load()
  
#   return jdbc_df

In [0]:
# Method for getting data from gcp 
from google.cloud import bigquery
from google.cloud import bigquery_storage
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType

def getGcpTable(query):
    # Initialize Spark Session
    spark = SparkSession.builder.getOrCreate()
    
    # Set up BigQuery clients
    client = bigquery.Client(project=project_id, credentials=credentials)
    bq_storage_client = bigquery_storage.BigQueryReadClient(credentials=credentials)

    # Execute the query and load data using BigQuery Storage API
    df = client.query(query).to_dataframe(bqstorage_client=bq_storage_client)

    # Convert to Spark DataFrame
    if df.empty:
        schema = StructType([])
        spark_df = spark.createDataFrame(spark.sparkContext.emptyRDD(), schema)
    else:
        spark_df = spark.createDataFrame(df)

    return spark_df

  # getting data from gcp
def spark_read_bigquery_table(query):
    df = spark.read.format("bigquery") \
    .option("query",query) \
    .option("parentProject", project_id) \
    .option("viewsEnabled", "true")\
    .option("materializationDataset", 'udco_ds_mrch')\
    .load()
    return df


In [0]:
# Method to convert date columns in delta lake to string
import pyspark.sql.functions as F
def stringify_timestamps(df):
  return df.select(*[
    F.col(c).cast("string").alias(c) if t == "timestamp" else F.col(c).alias(c)
    for c, t in df.dtypes
])

In [0]:
#Method to Fetch CCpoint SQL tables and return the result as a spark dataframe

def getCCPTable(tableName, query=None):
  if query is None:
    query = "(select * from {0}.{1}) code".format(Ccp_Schema, tableName)
  else:
    query = query.format(Ccp_Schema)
    
  Ccp_df = spark.read.jdbc(url=ccpoint_jdbcUrl, table=query, properties=connectionProperties_CCP)
  
  return Ccp_df

In [0]:
#Method to Fetch ASQL tables and return the result as a spark dataframe

def getASQLTable(tableName, query=None):
    if query is None:
        
        query = "{0}.{1}".format(asql_schema, tableName)
    else:
        query = query.format(asql_schema)   
    asql_df = spark.read.jdbc(url=asql_jdbcUrl, table=query, properties=connectionProperties_ASQL)  
    return asql_df

In [0]:
#Method to write data to ASQL Audting table
def addASQLTable(tableName, dataFrame, writemode):
    dataFrame.write.jdbc(asql_jdbcUrl,"{0}.{1}".format(asql_schema, tableName), mode=writemode ,properties=connectionProperties_ASQL)

In [0]:
# Method to insert data into DB2 table
def bulkinsertTable(tableName , df_data, writemode):
  try:
    
    tableName = tableName.format(db_schema)
    df_data.write.format ( "jdbc" ) \
      .option ( "driver" ,'com.ibm.db2.jcc.DB2Driver') \
      .option ( "url" , connection_string) \
      .option ( "dbtable" , tableName) \
      .mode(writemode).save()

  except Exception as e:       
    raise (e)

In [0]:
%scala
def bulkdeleteTable(query:String) : String = {
  
 
  val c=db_schema_db2_scl
  val query_scm=query.format(c)
   
  val connection = DriverManager.getConnection(jdbcUrl, jdbcUsername, jdbcPassword)
  val stmt = connection.createStatement()
  val rs = stmt.executeUpdate(query_scm) 
  connection.close() 
  return "Deleted"
  
   }

In [0]:
#Method to execute query against EDM with passphrase (Using PYTHON - Connection)

# def execEdm(query):    
#   query = query.format(edmDatabase_lwsn,edmDatabase)
#   con_session = snowflake.connector.connect(
#     user=passphrase_edm_user,
#     account=edmserver_py,
#     private_key=edm_private_key,
#     warehouse=edmWarehouse,
#     database=edmDatabase,
#     schema=edmSchema
#     )
#   cs = con_session.cursor()
#   cs.execute(query)
#   cs.close()
#   compl="Execution Completed"
#   return compl

In [0]:
#Method to execute query against EDM with passphrase (Using Snowflake Utils)

# def execEdmUtil(query):
    
#     snowquery = query.format(edmDatabase_lwsn,edmDatabase)
#     Snow_Utils = sc._jvm.net.snowflake.spark.snowflake.Utils
#     Snow_Utils.runQuery(sf_snow_pass_Options, snowquery)
    
#     compl="Execution Completed"
#     return compl

In [0]:
#Method to send email with SMTP

def sendMail(recipients,subject,mail_body,attachment_path=None,attachment_name=None):
  
  Status="Email Sent" 
  toaddr=recipients
  
  
  if attachment_path is None:
          
    msg = MIMEMultipart()
    msg['From'] = fromaddr
    msg['To'] = ", ".join(toaddr)
    msg['Subject'] = subject
    body = mail_body

    msg.attach(MIMEText(body, 'plain'))

    server = smtplib.SMTP(SMTP_Server, SMTP_Port)
    text = msg.as_string()
    server.sendmail(fromaddr, toaddr, text)
    server.quit()    
  
  else :
       
    msg = MIMEMultipart()
    msg['From'] = fromaddr
    msg['To'] = ", ".join(toaddr)
    msg['Subject'] = subject
    body = mail_body

    msg.attach(MIMEText(body, 'plain'))
    filename = attachment_name
    attachment = open(attachment_path, "rb")

    part = MIMEBase('application', 'octet-stream')
    part.set_payload((attachment).read())
    encoders.encode_base64(part)
    part.add_header('Content-Disposition', "attachment; filename= %s" % filename)
    msg.attach(part)
    server = smtplib.SMTP(SMTP_Server, SMTP_Port)
    text = msg.as_string()
    server.sendmail(fromaddr, toaddr, text)
    server.quit()  
         
  return Status

In [0]:
# Method for SFTP File Trasnsfer

def triggerSFTP(sftp_mode,local_path,remote_path):
  localpath = local_path
  remotepath = remote_path
  mode = sftp_mode
  client = paramiko.SSHClient()
  client.load_system_host_keys()
  client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
  client.connect(hostname=SFTP_Endpoint,username =SFTP_User,password=SFTP_Password)
  if mode=="push":
    sftp = client.open_sftp()
    sftp.put(localpath, remotepath)
    client.close()
  elif mode=="pull":
    sftp = client.open_sftp()
    sftp.get(remotepath,localpath)
    client.close()
  else :
    #print("Error ! Incorrect mode")
    dbutils.notebook.exit("Error ! Incorrect mode")

In [0]:
# Method for Fileshare Transfer

from azure.storage.fileshare import ShareFileClient

def fileShareTransfer(source_path,filshare_path,mode):
  try:
    stmt = ''
    local_path_start_pos = source_path.rfind('/')

    file_name = source_path [local_path_start_pos + 1:]
  
    file_client = ShareFileClient.from_connection_string(conn_str=Fileshare_con_string, share_name=filshare_path, file_path = file_name)
  

    if mode == "push" :
      with open(source_path, "rb") as source_file:
          file_client.upload_file(source_file)
          print("File "+source_path+" Uploaded to "+filshare_path+" in Fileshare")
    elif mode == "pull": 
      with open(source_path, "wb") as file_handle:
          data = file_client.download_file()
          data.readinto(file_handle)
          
      read_local_path = source_path [5:]
    else:
      dbutils.notebook.exit("Error ! Incorrect mode")
  except Exception as e:
    stmt = "Failure "+ str(e)
    raise (e)
  return stmt

In [0]:
%scala

//ASQL Execution method with Scala

def execASQL(query_asql:String) : String = {
  
  val query_asql_scm=query_asql.format(db_schema_asql_scl)
  
  val connectionProperties_asql = new Properties()
  connectionProperties_asql.put("user", s"${username_asql_scl}")
  connectionProperties_asql.put("password", s"${password_asql_scl}")
  connectionProperties_asql.setProperty("Driver", driverClass_asql_scl)
  val connection_asql = DriverManager.getConnection(jdbcUrl_asql_scl, username_asql_scl, password_asql_scl)
  val stmt_asql = connection_asql.createStatement()
  stmt_asql.execute(query_asql_scm)
  connection_asql.close()
  return "Executed"
  
   }

In [0]:
# # Method to Tarball & Compress directory 
# import tarfile
# import os.path
# def tarzipDirectory(output_filename, source_dir ,target_dir,clear_file_extention):
#     with tarfile.open(output_filename, "w:gz") as tar:
#         tar.add(source_dir, arcname=os.path.basename(source_dir))
#     driver_path="/databricks/driver/"+output_filename
#     dbutils.fs.cp("file:"+driver_path,target_dir)
#     os.remove(driver_path)
#     for folderName, subfolders, filenames in os.walk(source_dir):
#         for filename in filenames:
#             if filename.endswith("."+clear_file_extention):
#                 os.remove(source_dir+filename)
#     tar.close()

In [0]:
# Method to Tarball & Compress directory 
import tarfile
import os.path
def tarzipDirectory(output_filename, source_dir ,target_dir,clear_file_extention):
    with tarfile.open(output_filename, "w:gz") as tar:
            tar.add(source_dir, arcname=os.path.basename(source_dir))
    zip_path = os.path.join(os.getcwd(), output_filename)  # Get the full path to the zipped file
    dbutils.fs.cp("file:" + zip_path, target_dir)  # Copy the zipped file to the target directory in Azure Blob Storage
    print("Files zipped successfully!")
    os.remove(zip_path)
    for folderName, subfolders, filenames in os.walk(source_dir):
         for filename in filenames:
            if filename.endswith("."+clear_file_extention):
                os.remove(source_dir+filename)
    tar.close()

In [0]:
#Writting to BQ table
def addGCPTable(dataframe,tablename,writemode):
    gcp_tablename =f"gcp-abs-udco-mrbz-{gcpenv}-prj-01.udco_ds_space_planning.{tablename}"
    dataframe.write \
    .format("bigquery") \
    .option("table", gcp_tablename) \
    .option("parentProject", project_id) \
    .option("temporaryGcsBucket", bq_Bucket) \
    .mode(writemode) \
    .save()
 